# In-Memory Cache
> A batteries-included cache with TTL and size limits.

`InMemoryCacheBackend` provides a thread-safe, dictionary-backed cache with
configurable default TTL and maximum entry count.  It implements the
`CacheBackend` protocol and is ready to use out of the box.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.cache import InMemoryCacheBackend

## Create the Cache
Specify a default TTL (in seconds) and maximum number of entries.

In [ ]:
cache = InMemoryCacheBackend(default_ttl=300.0, max_size=1000)

print(f"Default TTL: {cache.default_ttl}s")
print(f"Max size:    {cache.max_size} entries")

## Store and Retrieve Values
`.set()` accepts any serializable value.  `.get()` returns `None` on a miss.

In [ ]:
# Store a dictionary
cache.set("key1", {"data": "value"})
print(f"Stored 'key1'")

# Store a list with a custom TTL
cache.set("key2", [1, 2, 3], ttl=60.0)
print(f"Stored 'key2' with 60s TTL")

# Retrieve
result = cache.get("key1")
print(f"\nGET 'key1': {result}")

# Miss
miss = cache.get("nonexistent")
print(f"GET 'nonexistent': {miss}")

## Delete Entries
Remove a single key with `.delete()`.

In [ ]:
deleted = cache.delete("key2")
print(f"Deleted 'key2': {deleted}")

# Confirm it's gone
print(f"GET 'key2' after delete: {cache.get('key2')}")

## Bulk Operations
Store multiple entries, then clear the entire cache.

In [ ]:
# Populate
for i in range(5):
    cache.set(f"item:{i}", f"value-{i}")
    print(f"  Stored 'item:{i}'")

# Verify one exists
print(f"\nGET 'item:3': {cache.get('item:3')}")

# Clear all
cache.clear()
print(f"\nAfter clear, GET 'item:3': {cache.get('item:3')}")

## Typical Usage Pattern
Cache expensive computations like embeddings or API responses.

In [ ]:
import hashlib


def compute_embedding(text: str) -> list:
    """Simulate an expensive embedding call."""
    print(f"  Computing embedding for: '{text[:30]}...'")
    return [0.1, 0.2, 0.3]  # mock


def get_embedding(text: str, cache: InMemoryCacheBackend) -> list:
    key = hashlib.sha256(text.encode()).hexdigest()[:16]
    cached = cache.get(key)
    if cached is not None:
        print(f"  Cache HIT for '{text[:30]}...'")
        return cached
    result = compute_embedding(text)
    cache.set(key, result)
    return result


embed_cache = InMemoryCacheBackend(default_ttl=600.0, max_size=500)

# First call: cache miss
print("Call 1:")
emb1 = get_embedding("What is machine learning?", embed_cache)

# Second call: cache hit
print("\nCall 2 (same input):")
emb2 = get_embedding("What is machine learning?", embed_cache)

print(f"\nSame result: {emb1 == emb2}")

## Key Takeaways
- `InMemoryCacheBackend` supports default TTL and max size
- `.set()` stores any value; `.get()` returns `None` on a miss
- `.delete()` removes one key; `.clear()` wipes everything
- Great for caching embeddings, API responses, and computed results

**Previous:** [Cache Backend Protocol](01_cache_backend.ipynb)